In [57]:
import pandas as pd
import re
import html
import nltk
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.corpus import stopwords
from nltk.tokenize import TweetTokenizer

nltk.download("stopwords")

# Keep negations because they can change the emotion expressed.
stop_words = set(stopwords.words("english")) - {"no", "nor", "not", "never", "cannot"}
tokenizer = TweetTokenizer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [58]:
from datasets import load_dataset

dataset = load_dataset("cardiffnlp/tweet_eval", "emotion")

train_df = dataset["train"].to_pandas()
val_df = dataset["validation"].to_pandas()
test_df = dataset["test"].to_pandas()

In [59]:
%pip install emoji contractions

Note: you may need to restart the kernel to use updated packages.


In [60]:
def clean_text(text):
    """
    Basic tweet preprocessing.
    """
    
    import contractions
    
    # Expand forms such as "don't" to "do not".
    text = contractions.fix(text)
    
    # Convert to lowercase
    text = html.unescape(text).lower()
    
    # Replace URLs
    text = re.sub(r"http\S+|www\S+", "", text)
    
    # Replace user mentions
    text = re.sub(r"@\w+", "", text)
    
    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    import emoji

    # "I love this ❤️" -> "I love this red_heart"
    text = emoji.demojize(text, delimiters=(" ", " "))
    
    # Keep hashtag words, but remove the # symbol
    text = re.sub(r"#(\w+)", r"\1", text)
    
    # Remove punctuation and symbols while keeping words and numbers.
    text = re.sub(r"[^\w\s]|_", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    
    tokens = tokenizer.tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]
    
    return " ".join(tokens)

In [61]:
train_df["clean_text"] = train_df["text"].apply(clean_text)
val_df["clean_text"] = val_df["text"].apply(clean_text)
test_df["clean_text"] = test_df["text"].apply(clean_text)

In [62]:
comparison = train_df[["text", "clean_text"]].head(10)

for _, row in comparison.iterrows():
    print("Original:", row["text"])
    print("Cleaned: ", row["clean_text"])
    print("-" * 80)

Original: “Worry is a down payment on a problem you may never have'.  Joyce Meyer.  #motivation #leadership #worry
Cleaned:  worry payment problem may never joyce meyer motivation leadership worry
--------------------------------------------------------------------------------
Original: My roommate: it's okay that we can't spell because we have autocorrect. #terrible #firstworldprobs
Cleaned:  roommate okay cannot spell autocorrect terrible firstworldprobs
--------------------------------------------------------------------------------
Original: No but that's so cute. Atsu was probably shy about photos before but cherry helped her out uwu
Cleaned:  no cute atsu probably shy photos cherry helped uwu
--------------------------------------------------------------------------------
Original: Rooneys fucking untouchable isn't he? Been fucking dreadful again, depay has looked decent(ish)tonight
Cleaned:  rooneys fucking untouchable not fucking dreadful depay looked decent ish tonight
-------

In [63]:
def clean_text_with_stopwords(text):
    return clean_text(text)

In [64]:
import os

os.makedirs("../data/processed", exist_ok=True)

for split_name, df in {
    "train": train_df,
    "val": val_df,
    "test": test_df,
}.items():
    # Keep the label column last in each full processed file.
    ordered_columns = [column for column in df.columns if column != "label"] + ["label"]
    processed_df = df.loc[:, ordered_columns]

    processed_df.to_csv(
        f"../data/processed/{split_name}_processed.csv",
        index=False,
        encoding="utf-8"
    )

    # Save a compact version for modeling with only cleaned text and its label.
    processed_df[["clean_text", "label"]].to_csv(
        f"../data/processed/{split_name}_cleaned_labeled.csv",
        index=False,
        encoding="utf-8"
    )

In [65]:
train_df

,text,label,clean_text
0,“Worry is a down payment on a problem you may ...,2,worry payment problem may never joyce meyer mo...
1,My roommate: it's okay that we can't spell bec...,0,roommate okay cannot spell autocorrect terribl...
2,No but that's so cute. Atsu was probably shy a...,1,no cute atsu probably shy photos cherry helped...
3,Rooneys fucking untouchable isn't he? Been fuc...,0,rooneys fucking untouchable not fucking dreadf...
4,it's pretty depressing when u hit pan on ur fa...,3,pretty depressing hit pan favourite highlighter
...,...,...,...
3252,I get discouraged because I try for 5 fucking ...,3,get discouraged try 5 fucking years contact la...
3253,The @user are in contention and hosting @user ...,3,contention hosting nation camden empty sad
3254,@user @user @user @user @user as a fellow UP g...,0,fellow grad shiver shallowness arguments
3255,You have a #problem? Yes! Can you do #somethin...,0,problem yes something no
